In [1]:
# Calculate market features from prices and trading observations on or before prediction.
# Keep insufficient prelisting history missing.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from project_config import (
    MARKET_FEATURE_SOURCE_FIELDS,
    MARKET_HORIZONS_MONTHS as HORIZONS,
    MARKET_FEATURE_COLUMNS as FEATURE_COLUMNS,
)
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/trend_features.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
MAX_MARKET_AGE_DAYS = 45
MONTHS_PER_YEAR = 12
trend_features = pd.read_csv(PROCESSED / "trend_features.csv")
required = [
    "event_id",
    "company_id",
    "prediction_date",
    "adjusted_price",
    "market_adjusted_price_reference_date",
]
required += [f"has_{months}m_market_history" for months in HORIZONS]
missing = sorted(set(required) - set(trend_features.columns))
if trend_features.empty or missing:
    raise ValueError(f"Empty input or missing required columns: {missing}")
if (
    trend_features.duplicated(["company_id", "prediction_date"]).any()
    or not trend_features["event_id"].is_unique
):
    raise ValueError("Input must have unique company-events")
if set(FEATURE_COLUMNS) & set(trend_features.columns):
    raise ValueError("Input already contains these market feature names")
events = trend_features[["event_id", "company_id", "prediction_date"]].copy()
events["prediction_date"] = pd.to_datetime(
    events["prediction_date"], errors="raise"
).dt.normalize()
if events.isna().any().any():
    raise ValueError("Missing event keys or prediction dates")
chunks = []
company_ids = set(events["company_id"])
for chunk in pd.read_csv(
    PROCESSED / "clean_company_long.csv",
    usecols=["company_id", "date", "field", "data_type", "value"],
    chunksize=250_000,
):
    relevant = (
        chunk["company_id"].isin(company_ids)
        & chunk["data_type"].eq("market")
        & chunk["field"].isin(MARKET_FEATURE_SOURCE_FIELDS)
    )
    chunks.append(chunk.loc[relevant].copy())
market_long = pd.concat(chunks, ignore_index=True)
market_long["date"] = pd.to_datetime(market_long["date"], errors="raise").dt.normalize()
market_long["value"] = pd.to_numeric(market_long["value"], errors="raise")
if (
    market_long["date"].isna().any()
    or market_long.duplicated(["company_id", "date", "field"]).any()
):
    raise ValueError("Missing dates or duplicate market records")
if not np.isfinite(market_long.loc[market_long["value"].notna(), "value"]).all():
    raise ValueError("Nonfinite market source values; inspect cleaning")
available_source_fields = set(market_long["field"])
source_histories = {}
for (company, field), group in market_long.loc[market_long["value"].notna()].groupby(
    ["company_id", "field"], sort=False
):
    group = group.sort_values("date")
    if field == "adjusted_price":
        group = group.loc[group["value"].gt(0)]
    source_histories[(company, field)] = (
        group["date"].to_numpy(dtype="datetime64[ns]"),
        group["value"].to_numpy(dtype=float),
    )

In [3]:
# Endpoint selection uses the latest observation on or before a specified cutoff.
def endpoint(dates, values, cutoff, max_age_days):
    position = np.searchsorted(dates, cutoff, side="right") - 1
    if position < 0 or dates[position] < cutoff - np.timedelta64(max_age_days, "D"):
        return np.nan, np.datetime64("NaT", "ns")
    return float(values[position]), dates[position]


def trailing_monthly_window(dates, values, prediction, observations, max_age_days):
    stop = np.searchsorted(dates, prediction, side="right")
    if not stop or dates[stop - 1] < prediction - np.timedelta64(max_age_days, "D"):
        return np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)
    before = dates[:stop]
    months = before.astype("datetime64[M]").astype(np.int64)
    positions = np.flatnonzero(np.r_[months[:-1] != months[1:], True])[-observations:]
    if len(positions) != observations or not (np.diff(months[positions]) == 1).all():
        return np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)
    return dates[positions], values[positions]


def price_path_statistics(prices):
    if len(prices) < 3 or not np.isfinite(prices).all() or not (prices > 0).all():
        return np.nan, np.nan
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        returns = prices[1:] / prices[:-1] - 1
        volatility = np.std(returns, ddof=1) * np.sqrt(MONTHS_PER_YEAR)
        drawdowns = prices / np.maximum.accumulate(prices) - 1
        maximum_drawdown = np.min(drawdowns)
    if not np.isfinite(returns).all() or not np.isfinite(volatility):
        volatility = np.nan
    if not np.isfinite(maximum_drawdown):
        maximum_drawdown = np.nan
    return volatility, maximum_drawdown


def safe_scalar_divide(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0:
        return np.nan
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        result = np.divide(numerator, denominator)
    return float(result) if np.isfinite(result) else np.nan

In [4]:
# Calculate every feature from the company's cleaned market histories.
def calculate_market_features(event_rows, histories, horizons, max_age_days):
    records = []
    audit_records = []
    empty_dates = np.array([], dtype="datetime64[ns]")
    empty_values = np.array([], dtype=float)
    for event in event_rows.itertuples(index=False):
        prediction = event.prediction_date.to_datetime64()
        record = {column: np.nan for column in FEATURE_COLUMNS}
        audit = {}
        price_dates, price_values = histories.get(
            (event.company_id, "adjusted_price"), (empty_dates, empty_values)
        )
        current_price, current_date = endpoint(
            price_dates, price_values, prediction, max_age_days
        )
        audit["market_feature_price_reference_date"] = current_date
        for horizon in horizons:
            historical_cutoff = (
                event.prediction_date - pd.DateOffset(months=horizon)
            ).to_datetime64()
            baseline_price, baseline_date = endpoint(
                price_dates, price_values, historical_cutoff, max_age_days
            )
            audit[f"return_{horizon}m_baseline_cutoff_date"] = historical_cutoff
            window_dates, window_prices = trailing_monthly_window(
                price_dates, price_values, prediction, horizon + 1, max_age_days
            )
            sufficient_history = len(window_prices) == horizon + 1
            audit[f"market_price_window_start_date_{horizon}m"] = (
                window_dates[0] if sufficient_history else np.datetime64("NaT", "ns")
            )
            audit[f"market_price_window_end_date_{horizon}m"] = (
                window_dates[-1] if sufficient_history else np.datetime64("NaT", "ns")
            )
            audit[f"market_price_window_n_prices_{horizon}m"] = len(window_prices)
            audit[f"market_price_window_n_returns_{horizon}m"] = max(
                0, len(window_prices) - 1
            )
            # Return endpoints follow calendar prediction offsets. Complete monthly
            # history is still required; endpoints alone cannot certify a full path.
            audit[f"return_{horizon}m_baseline_date"] = np.datetime64("NaT", "ns")
            if sufficient_history:
                ratio = safe_scalar_divide(current_price, baseline_price)
                if np.isfinite(ratio):
                    record[f"return_{horizon}m"] = ratio - 1
                    audit[f"return_{horizon}m_baseline_date"] = baseline_date
                volatility, drawdown = price_path_statistics(window_prices)
                record[f"volatility_{horizon}m"] = volatility
                record[f"maximum_drawdown_{horizon}m"] = drawdown
        one_year_before = (
            event.prediction_date - pd.DateOffset(months=12)
        ).to_datetime64()
        for field in ["market_cap", "volume", "turnover"]:
            dates, values = histories.get(
                (event.company_id, field), (empty_dates, empty_values)
            )
            current, current_field_date = endpoint(
                dates, values, prediction, max_age_days
            )
            previous, previous_field_date = endpoint(
                dates, values, one_year_before, max_age_days
            )
            audit[f"{field}_change_12m_current_date"] = np.datetime64("NaT", "ns")
            audit[f"{field}_change_12m_baseline_date"] = np.datetime64("NaT", "ns")
            if np.isfinite(current) and np.isfinite(previous):
                with np.errstate(over="ignore", invalid="ignore"):
                    change = np.subtract(current, previous)
                if np.isfinite(change):
                    record[f"{field}_change_12m"] = float(change)
                    audit[f"{field}_change_12m_current_date"] = current_field_date
                    audit[f"{field}_change_12m_baseline_date"] = previous_field_date
        spread_dates, spread_values = histories.get(
            (event.company_id, "bid_ask_spread"), (empty_dates, empty_values)
        )
        dates, spreads = trailing_monthly_window(
            spread_dates, spread_values, prediction, 12, max_age_days
        )
        audit["bid_ask_spread_average_12m_start_date"] = np.datetime64("NaT", "ns")
        audit["bid_ask_spread_average_12m_end_date"] = np.datetime64("NaT", "ns")
        audit["bid_ask_spread_average_12m_n_obs"] = len(spreads)
        if len(spreads) == 12:
            with np.errstate(over="ignore", invalid="ignore"):
                average = np.mean(spreads)
            if np.isfinite(average):
                record["bid_ask_spread_average_12m"] = float(average)
                audit["bid_ask_spread_average_12m_start_date"] = dates[0]
                audit["bid_ask_spread_average_12m_end_date"] = dates[-1]
        records.append(record)
        audit_records.append(audit)
    features = pd.DataFrame(records, index=event_rows.index, columns=FEATURE_COLUMNS)
    audits = pd.DataFrame(audit_records, index=event_rows.index)
    for column in audits:
        if "_date" in column:
            audits[column] = (
                pd.to_datetime(audits[column]).dt.normalize().astype("datetime64[ns]")
            )
    return features, audits


market_feature_values, market_feature_audit = calculate_market_features(
    events, source_histories, HORIZONS, MAX_MARKET_AGE_DAYS
)
market_feature_values["stock_return_12m"] = market_feature_values["return_12m"]
market_features = pd.concat(
    [trend_features.copy(), market_feature_values, market_feature_audit], axis=1
)
feature_summary = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "available_events": [
            int(market_features[column].notna().sum()) for column in FEATURE_COLUMNS
        ],
        "missing_events": [
            int(market_features[column].isna().sum()) for column in FEATURE_COLUMNS
        ],
        "missing_pct": [
            100 * market_features[column].isna().mean() for column in FEATURE_COLUMNS
        ],
    }
)

In [5]:
# Temporal checks include every endpoint and every sampled price-window boundary.
for column in market_feature_audit:
    if "_date" in column:
        source_dates = market_feature_audit[column]
        observed = source_dates.notna()
reference_dates = market_feature_audit["market_feature_price_reference_date"]
for horizon in HORIZONS:
    history_flag = trend_features[f"has_{horizon}m_market_history"].astype(bool)
    complete = market_feature_audit[f"market_price_window_n_returns_{horizon}m"].eq(
        horizon
    )
    observed_return = market_features[f"return_{horizon}m"].notna()
    baseline = market_feature_audit[f"return_{horizon}m_baseline_date"]
    cutoff = market_feature_audit[f"return_{horizon}m_baseline_cutoff_date"]
    volatility = market_features[f"volatility_{horizon}m"]
    drawdown = market_features[f"maximum_drawdown_{horizon}m"]
for field in ["market_cap", "volume", "turnover"]:
    available = market_features[f"{field}_change_12m"].notna()
    current_date = market_feature_audit[f"{field}_change_12m_current_date"]
    baseline_date = market_feature_audit[f"{field}_change_12m_baseline_date"]
    cutoff = events["prediction_date"] - pd.DateOffset(months=12)

output_path = PROCESSED / "market_features.csv"
market_features.to_csv(output_path, index=False, date_format="%Y-%m-%d")